# Bridge - Patrón Estructural

## Introducción

El patrón **Bridge** separa una abstracción de su implementación para que ambas puedan variar de forma independiente. En este ejemplo, la abstracción representa operaciones de pago de una tienda y la implementación representa los mecanismos concretos que procesan el dinero.

## Objetivos

- Identificar el problema de crear una clase por cada combinación de operación y medio de pago.
- Separar la jerarquía de operaciones de pago de la jerarquía de procesadores.
- Combinar cualquier operación con cualquier procesador sin duplicar clases.

## Contexto

Una tienda necesita procesar pagos mediante diferentes medios, como tarjeta de crédito y transferencia bancaria. Cada medio de pago tiene una API o mecanismo interno distinto para retirar y recibir dinero.

## Problema

Sin una separación entre la operación de pago y el mecanismo que realmente mueve el dinero, la clase de la tienda tendría que conocer los detalles de cada medio de pago. Al agregar nuevos medios o cambiar la forma de procesarlos, el código crecería y se volvería difícil de mantener.

Por ejemplo, una solución directa puede crear una clase distinta para cada combinación entre una operación de la tienda y un mecanismo de pago. Si aparecen más operaciones o medios, aumentan las clases y se repite la lógica.

En la primera parte se muestra una solución directa, sin patrón. En la segunda se aplica **Bridge** para separar:

- **Abstracción:** la operación de alto nivel que realiza el pago.
- **Implementación:** el mecanismo concreto que procesa el dinero.

Así, la abstracción depende de una interfaz de procesamiento y no de una clase concreta.

## Sin patrón Bridge (forma directa)

In [1]:
class CompraConTarjeta:
    def pagar(self, monto):
        print(f"Compra con tarjeta por ${monto} procesada")


class CompraConTransferencia:
    def pagar(self, monto):
        print(f"Compra con transferencia por ${monto} procesada")


class SuscripcionConTarjeta:
    def pagar(self, monto):
        print(f"Suscripción con tarjeta por ${monto} procesada")


class SuscripcionConTransferencia:
    def pagar(self, monto):
        print(f"Suscripción con transferencia por ${monto} procesada")


# Cada nueva operación debe repetirse para cada medio de pago.
CompraConTarjeta().pagar(50000)
CompraConTransferencia().pagar(50000)
SuscripcionConTarjeta().pagar(30000)
SuscripcionConTransferencia().pagar(30000)

Compra con tarjeta por $50000 procesada
Compra con transferencia por $50000 procesada
Suscripción con tarjeta por $30000 procesada
Suscripción con transferencia por $30000 procesada


## Con el patrón Bridge implementado

La solución separa dos jerarquías:

- **Abstracción:** `MetodoPago`, con las abstracciones refinadas `Compra` y `Suscripcion`.
- **Implementación:** `ProcesadorPago`, con los implementadores concretos `ProcesadorTarjeta` y `ProcesadorTransferencia`.

`MetodoPago` mantiene una referencia a `ProcesadorPago` mediante composición. Así, una compra o una suscripción puede utilizar cualquier procesador sin crear una clase específica para cada combinación.

Por ejemplo, `Compra(ProcesadorTarjeta())` y `Compra(ProcesadorTransferencia())` representan la misma operación con implementaciones diferentes. Si se agrega otra operación o un nuevo procesador, solo se crea una nueva clase en la jerarquía correspondiente.

In [2]:
from abc import ABC, abstractmethod


# Implementador: define las operaciones que todos los procesadores deben ofrecer.
class ProcesadorPago(ABC):
    @abstractmethod
    def procesar(self, monto, concepto):
        pass


# Implementadores concretos.
class ProcesadorTarjeta(ProcesadorPago):
    def procesar(self, monto, concepto):
        return f"{concepto} de ${monto} procesada con tarjeta de crédito"


class ProcesadorTransferencia(ProcesadorPago):
    def procesar(self, monto, concepto):
        return f"{concepto} de ${monto} procesada mediante transferencia bancaria"


# Abstracción: mantiene el puente hacia el procesador.
class MetodoPago(ABC):
    def __init__(self, procesador):
        self.procesador = procesador

    @abstractmethod
    def obtener_concepto(self):
        pass

    def pagar(self, monto):
        concepto = self.obtener_concepto()
        return self.procesador.procesar(monto, concepto)


# Abstracciones refinadas: agregan tipos de operación sin conocer los procesadores.
class Compra(MetodoPago):
    def obtener_concepto(self):
        return "Compra"


class Suscripcion(MetodoPago):
    def obtener_concepto(self):
        return "Suscripción"


# La misma abstracción puede combinarse con implementaciones diferentes.
compra_tarjeta = Compra(ProcesadorTarjeta())
compra_transferencia = Compra(ProcesadorTransferencia())
suscripcion_tarjeta = Suscripcion(ProcesadorTarjeta())
suscripcion_transferencia = Suscripcion(ProcesadorTransferencia())

print(compra_tarjeta.pagar(50000))
print(compra_transferencia.pagar(50000))
print(suscripcion_tarjeta.pagar(30000))
print(suscripcion_transferencia.pagar(30000))

Compra de $50000 procesada con tarjeta de crédito
Compra de $50000 procesada mediante transferencia bancaria
Suscripción de $30000 procesada con tarjeta de crédito
Suscripción de $30000 procesada mediante transferencia bancaria


## UML del patrón Bridge

```plantuml
@startuml
abstract class MetodoPago {
    - procesador: ProcesadorPago
    + pagar(monto)
}

class Compra
class Suscripcion

abstract class ProcesadorPago {
    + procesar(monto, concepto)
}

class ProcesadorTarjeta
class ProcesadorTransferencia

MetodoPago o-- ProcesadorPago
MetodoPago <|-- Compra
MetodoPago <|-- Suscripcion
ProcesadorPago <|-- ProcesadorTarjeta
ProcesadorPago <|-- ProcesadorTransferencia
@enduml
```

## ¿Por qué Bridge y no otro patrón?

Se eligió **Bridge** porque existen dos dimensiones que pueden cambiar independientemente: el tipo de operación (`Compra` o `Suscripcion`) y el mecanismo de procesamiento (`ProcesadorTarjeta` o `ProcesadorTransferencia`). El patrón evita crear una clase para cada combinación.

**Strategy** también permite intercambiar algoritmos, pero aquí el objetivo principal no es seleccionar un algoritmo para una operación fija. Es separar dos jerarquías completas y permitir que sus clases se combinen mediante composición; por eso Bridge representa mejor el problema.